# Lesson 2 — Choice: picking one option from a fixed set

Lesson 1 covered what a System One model is and what `state` means. This lesson covers the first primitive — `Choice` — and you'll run a real question against TypeSafe's `Jev` model for the first time.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/primitives/choice (click to expand)</summary>

# Choice

> A Choice is a System One question type for selecting one option from a defined set. The answer includes the selected option, a probability for each option, and confidence.

Use a Choice when the answer is one of a fixed set of options. For example, which team handles a ticket, which category a product belongs to, or which language a code snippet is written in. If the answer is a position on a spectrum, use a [Score](https://docs.typesafe.ai/primitives/score). If it's a yes or no, use a [Noul](https://docs.typesafe.ai/primitives/noul).

A Choice answer is the selected option in `choice`. The model also returns a probability for every option in `probabilities`, and a `confidence` value for the selected option.

Example questions:

```
"What programming language is this code written in"
  → options: python, javascript, typescript, go, rust, other

"What type of meeting is this based on the title and description"
  → options: standup, planning, retrospective, one on one, brainstorm, none of the above

"Which product category does this item belong to"
  → options: electronics, clothing, home garden, food and beverage
```

## Request structure

The top level has three fields: `state`, the content to evaluate; `model`; and `questions`, a map from question ids you choose to question objects. Each Choice question has:

- `type`: Always `"choice"`.
- `instructions`: The question the model answers.
- `criteria`: The answer options, as a map. Each key is an option name and each value is a description of that option.

```json
{
  "state": "My running shoes arrived in the wrong size. Can I swap them for a size 10?",
  "questions": {
    "department": {
      "type": "choice",
      "instructions": "Which team should handle this?",
      "criteria": {
        "returns": "Exchanges, refunds, wrong or damaged items",
        "shipping": "Delivery status, delays, lost packages",
        "billing": "Charges, invoices, payment problems"
      }
    }
  }
}
```

You choose the question id, `department` in this case. The model never sees the question id. The option names and their descriptions are both sent to the model, so write descriptions that separate the options from each other.

## Response structure

```json
{
  "model": "jev-latest",
  "answers": {
    "department": {
      "type": "choice",
      "choice": "returns",
      "confidence": 1.0,
      "probabilities": {
        "shipping": 0.0,
        "returns": 1.0,
        "billing": 0.0
      }
    }
  },
  "usage": {
    "input_tokens": 330,
    "output_tokens": 34
  }
}
```

Besides `type`, each Choice answer has three values:

- `choice`: The option with the highest probability.
- `probabilities`: The full probability distribution across every option. The sum of all values is 1.
- [`confidence`](https://docs.typesafe.ai/confidence): A number from 0 to 1 computed from how `probabilities` is spread. A flat shape, with probability spread across several options, means low confidence. A single peak on one option means high confidence.

This ticket is an easy one, so all of the probability is on `returns` and confidence is 1.0. A ticket that mentions a wrong size and a missing refund would split probability between `returns` and `billing`, and confidence would drop.

## Good practice: ask more than one question per call

Ask every Choice question your code might need in a single request rather than one request per question. Questions are evaluated in parallel. Adding questions barely changes the response time, and the code can ignore answers it doesn't need. Extra questions still cost tokens.

The same logic applies to the options inside a single Choice question. A Choice question accepts up to 255 options, and adding options costs a few tokens each, so give the model the full list of teams, categories, or products rather than a shortlist. Add an `other` or `none of the above` option when the list might not cover every input.

To classify documents through a deep hierarchy or large taxonomy, chain Choice questions level by level — the [Hierarchical Classification cookbook](https://docs.typesafe.ai/cookbooks/hierarchical_classification) shows how to run a beam search over Choice probabilities.

> This page also walks through a five-question ticket-triage example with a full Python routing function, and a section on structuring `instructions` and `criteria` as JSON objects instead of plain strings. The routing example is trimmed here since it mostly repeats speculative fan-out (covered later); structured instructions get their own dedicated lesson. Read the full page at [docs.typesafe.ai/primitives/choice](https://docs.typesafe.ai/primitives/choice) any time.

</details>

## The Choice primitive

Alongside `state`, every call needs one or more **questions** — each one typed. `Choice` is the first: pick exactly one option from a fixed set you define. The model returns the picked option, a probability for every option, and a confidence score.

Let's ask Jev to categorize a support ticket. This is TypeSafe's own canonical example (from its docs and JS SDK).

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { choice } from "npm:@typesafe-ai/sdk@^0.7.0";

const response = await client.systemOne({
  state: { document: "I found I have a credit issue with my account. Please fix this ASAP." },
  questions: {
    category: choice("What is this ticket about?", {
      billing: null,
      technical: null,
      other: null,
    }),
  },
});

console.log(response.answers.category.choice);

## What just happened

`response.answers.category.choice` is the model's pick — one of exactly `billing`, `technical`, or `other`. There's no text to parse, and the answer literally cannot be a fourth option: the *shape* of the answer is guaranteed by construction. (Whether it picked the *correct* one is a different question — one this series comes back to.)

**Try it yourself:** change the `document` text in the cell above to something clearly technical, or something ambiguous. Before you re-run the cell, guess what `category` will come back as — then check.

## Next up

Next lesson: `Score` — for when the answer isn't one option, but a position along a spectrum.

Questions? Ask your agent — it has the same research this lesson was built from, in `research/typesafe-ai.md`.